### SWDA rebalancing

The purpose of this notebook is to evaluate a rebalancing of a stock portfolio having SWDA etf. The purpose is to find the weight of the XMMI etf that makes it as close to VWCE as possible

In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from scipy.optimize import minimize
from portfolio_balancing.swda_xls_to_csv import *
from pathlib import Path
import requests
from datetime import datetime
import openpyxl

In [2]:
# Download SWDA allocation file from iShares website

url = (
    "https://www.ishares.com/uk/individual/en/products/251882/ishares-msci-world-ucits-etf-acc-fund/1535604580409.ajax?fileType=xls&fileName=iShares-Core-MSCI-World-UCITS-ETF_fund&dataType=fund"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.ishares.com/uk/individual/en/products/251882/",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

response = requests.get(url, headers=headers, stream=True)
response.raise_for_status()

# Nome file con data odierna
today = datetime.today().strftime("%Y-%m-%d")
filename = "portfolio_allocations/SWDA_allocation.xls"

with open(filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"File salvato: {filename}")


File salvato: portfolio_allocations/SWDA_allocation.xls


In [3]:
# VWCE is different: it doesn't have a direct excel download but instead i have to convert a json file to xlsx

url = "https://www.nl.vanguard/gpx/graphql"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Origin": "https://www.nl.vanguard",
    "Referer": "https://www.nl.vanguard/professional/product/etf/equity/9679/ftse-all-world-ucits-etf-usd-accumulating",
    "apollographql-client-name": "gpx",
    "x-consumer-id": "nl0",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
}

payload = {
    "operationName": "MarketAllocationGqlQuery",
    "variables": {"portIds": ["9679"]},
    "query": """query MarketAllocationGqlQuery($portIds: [String!]!) {
  funds(portIds: $portIds) {
    profile {
      fundFullName
      primaryMarketEquityClassification
      polarisPdtTypeIndicator
      marketOfDomicile
      __typename
    }
    marketAllocation {
      portId
      date
      countryCode
      countryName
      fundMktPercent
      holdingStatCode
      benchmarkMktPercent
      regionCode
      regionName
      __typename
    }
    __typename
  }
}
"""
}

response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()

data = response.json()
allocations = data["data"]["funds"][0]["marketAllocation"]

# Tieni solo i dati FTSE (esclude duplicati MSCI)
allocations = [a for a in allocations if a["holdingStatCode"].startswith("FT")]


# Salva in Excel
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Market Allocation"

ws.append(["countryCode", "countryName", "regionCode", "regionName",
           "fundMktPercent", "benchmarkMktPercent", "date"])

for item in allocations:
    ws.append([
        item["countryCode"],
        item["countryName"],
        item["regionCode"],
        item["regionName"],
        item["fundMktPercent"],
        item["benchmarkMktPercent"],
        item["date"],
    ])

filename = "portfolio_allocations/VWCE_allocation.xlsx"
wb.save(filename)
print(f"File salvato: {filename} ({len(allocations)} paesi)")

File salvato: portfolio_allocations/VWCE_allocation.xlsx (50 paesi)


In [4]:
# Download XMME allocation file from iShares website

url = (
    "https://etf.dws.com/etfdata/export/GBR/ENG/excel/product/constituent/IE00BTJRMP35/"
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Accept-Language": "it-IT,it;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://etf.dws.com/",
}

response = requests.get(url, headers=headers, stream=True)
response.raise_for_status()

# Nome file con data odierna
today = datetime.today().strftime("%Y-%m-%d")
filename = "portfolio_allocations/XMME_allocation.xlsx"

with open(filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"File salvato: {filename}")





File salvato: portfolio_allocations/XMME_allocation.xlsx


In [5]:
# convert SWDA allocation file from xls to csv
input_path = Path('portfolio_allocations/SWDA_allocation.xls')
output_path = Path('portfolio_allocations/SWDA_allocation.csv')

convert(input_path, output_path)



1361

In [7]:
#get portfolio geographical info for SWDA
SWDA_alloc = pd.read_csv('portfolio_allocations/SWDA_allocation.csv')

SWDA_alloc = SWDA_alloc.rename(columns={'Weight (%)': 'Weight'})
SWDA_alloc_geo = SWDA_alloc.groupby('Location')['Weight'].sum()
SWDA_alloc_geo = SWDA_alloc_geo.rename(index = {'--': 'Other'})
SWDA_alloc_geo

swda_countries = SWDA_alloc_geo.sort_index().index.tolist()

In [8]:
# get portfolio geographical info for XMME
df_temp = pd.read_excel('portfolio_allocations\XMME_allocation.xlsx', header=None)

header_row_index = df_temp.index[df_temp.eq("Name").any(axis=1)].tolist()[0]

XMME_alloc = pd.read_excel('portfolio_allocations\XMME_allocation.xlsx', skiprows=header_row_index, index_col = 0)

XMME_alloc['Weighting'] = XMME_alloc['Weighting'].astype(float) * 100
XMME_alloc = XMME_alloc.rename(columns={'Weighting': 'Weight (%)'})
XMME_alloc_geo = XMME_alloc.groupby('Country')['Weight (%)'].sum()

XMME_alloc_geo = XMME_alloc_geo.rename(index = {'-': 'Other'})
xmme_countries = XMME_alloc_geo.sort_index().index.tolist()

<>:2: SyntaxWarning: invalid escape sequence '\X'
<>:6: SyntaxWarning: invalid escape sequence '\X'
<>:2: SyntaxWarning: invalid escape sequence '\X'
<>:6: SyntaxWarning: invalid escape sequence '\X'
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\4000533395.py:2: SyntaxWarning: invalid escape sequence '\X'
  df_temp = pd.read_excel('portfolio_allocations\XMME_allocation.xlsx', header=None)
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\4000533395.py:6: SyntaxWarning: invalid escape sequence '\X'
  XMME_alloc = pd.read_excel('portfolio_allocations\XMME_allocation.xlsx', skiprows=header_row_index, index_col = 0)


In [9]:
XMME_alloc_geo

Country
Other                    0.262247
Australia                0.040265
Brazil                   4.848324
Canada                   0.046794
Chile                    0.530978
China                   23.466546
Colombia                 0.161581
Czech Republic           0.128831
Egypt                    0.055904
Greece                   0.509571
Hong Kong                0.164098
Hungary                  0.338866
India                   12.558125
Indonesia                0.851885
Ireland                  0.040960
Kazakhstan               0.019383
Korea, Republic of      16.983031
Kuwait                   0.615236
Luxembourg               0.063152
Malaysia                 1.170131
Mexico                   2.183229
Netherlands              0.066072
Peru                     0.268919
Philippines              0.345937
Poland                   1.113351
Qatar                    0.589001
Romania                  0.045321
Russian Federation       0.000002
Saudi Arabia             2.812334
South 

In [ ]:
#get portfolio geographical info for VWCE -> it already is allocated by Country


VWCE_alloc_geo = pd.read_excel('portfolio_allocations\VWCE_allocation.xlsx', header = 0)
VWCE_alloc_geo = VWCE_alloc_geo.rename(columns={'fundMktPercent': 'Weight'})
VWCE_alloc_geo = VWCE_alloc_geo.reset_index().set_index('countryName')['Weight']
vwce_countries = VWCE_alloc_geo.sort_index().index.to_list()
VWCE_alloc_geo.index.name = 'Country'


<>:4: SyntaxWarning: invalid escape sequence '\V'
<>:4: SyntaxWarning: invalid escape sequence '\V'
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\1493966414.py:4: SyntaxWarning: invalid escape sequence '\V'
  VWCE_alloc_geo = pd.read_excel('portfolio_allocations\VWCE_allocation.xlsx', header = 0)


In [11]:
VWCE_alloc_geo

,countryCode,countryName,regionCode,regionName,fundMktPercent,benchmarkMktPercent,date
0,AE,United Arab Emirates,EM,Emerging Markets,0.19633,0.19402,2026-02-28
1,AT,Austria,EU,Europe,0.07527,0.07572,2026-02-28
2,AU,Australia,FE,Pacific,1.73577,1.73688,2026-02-28
3,BE,Belgium,EU,Europe,0.26384,0.25382,2026-02-28
4,BR,Brazil,EM,Emerging Markets,0.51096,0.50919,2026-02-28
5,CA,Canada,NaN,North America,3.07141,3.06726,2026-02-28
6,CH,Switzerland,EU,Europe,2.24053,2.24044,2026-02-28
7,CL,Chile,EM,Emerging Markets,0.07728,0.07280,2026-02-28
8,CN,China,EM,Emerging Markets,3.15149,3.17367,2026-02-28
9,CO,Colombia,EM,Emerging Markets,0.01383,0.01672,2026-02-28


In [13]:
VWCE_alloc_geo

Country
United Arab Emirates     0.19633
Austria                  0.07527
Australia                1.73577
Belgium                  0.26384
Brazil                   0.51096
Canada                   3.07141
Switzerland              2.24053
Chile                    0.07728
China                    3.15149
Colombia                 0.01383
Czech Republic           0.01433
Germany                  2.08922
Denmark                  0.34528
Egypt                    0.00598
Spain                    0.86433
Finland                  0.26286
France                   2.13752
United Kingdom           3.59639
Greece                   0.08175
Hong Kong                0.53316
Hungary                  0.04141
Indonesia                0.11219
Ireland                  0.07982
Israel                   0.29601
India                    1.77487
Iceland                  0.01136
Italy                    0.77773
Japan                    6.29802
Korea                    2.22604
Kuwait                   0.07370
Me

In [14]:
vwce_countries = set(vwce_countries)
swda_countries = set(swda_countries)
xmme_countries = set(xmme_countries)

swda_xmme_countries = swda_countries.union(xmme_countries)

print('Countries in SWDA and XMME:', swda_xmme_countries)
print('Countries in VWCE and not in SWDA/XMME:', vwce_countries - swda_xmme_countries)     
print('Countries in SWDA/XMME and not in VWCE:', swda_xmme_countries - vwce_countries)
print('Countries in all three:', swda_xmme_countries.intersection(vwce_countries))


Countries in SWDA and XMME: {'Israel', 'Greece', 'Egypt', 'Austria', 'Indonesia', 'Saudi Arabia', 'United Kingdom', 'Italy', 'China', 'Australia', 'Czech Republic', 'United Arab Emirates', 'Hong Kong', 'Luxembourg', 'Thailand', 'Spain', 'Mexico', 'Finland', 'Poland', 'European Union', 'India', 'Sweden', 'Brazil', 'Ireland', 'Philippines', 'United States', 'Norway', 'Turkey', 'Canada', 'Germany', 'Singapore', 'Kuwait', 'Malaysia', 'Korea, Republic of', 'Portugal', 'Qatar', 'New Zealand', 'Romania', 'France', 'Russian Federation', 'Netherlands', 'Belgium', 'Chile', 'Switzerland', 'Taiwan', 'Kazakhstan', 'Colombia', 'Peru', 'South Africa', 'Denmark', 'Hungary', 'Other', 'Japan'}
Countries in VWCE and not in SWDA/XMME: {'Iceland', 'Russia', 'Korea'}
Countries in SWDA/XMME and not in VWCE: {'Luxembourg', 'Russian Federation', 'Kazakhstan', 'Peru', 'Korea, Republic of', 'European Union'}
Countries in all three: {'Israel', 'Greece', 'Indonesia', 'Austria', 'Egypt', 'Saudi Arabia', 'United Kin

In [ ]:
# Fix a couple of name ambiguities in country names
VWCE_alloc_geo = VWCE_alloc_geo.rename(index = {'United States of America': 'United States'})
XMME_alloc_geo = XMME_alloc_geo.rename(index = {'Korea, Republic of': 'South Korea'})

#There is still eteronimy in the way South Korea and Russia are named in the three datasets
if "South korea" in XMME_alloc_geo.index:
    XMME_alloc_geo = XMME_alloc_geo.rename(index = {'South korea': 'Korea'})
if "South Korea" in SWDA_alloc_geo.index:
    SWDA_alloc_geo.rename(index = {'South Korea': 'Korea'})
if "South korea" in VWCE_alloc_geo.index:
    VWCE_alloc_geo = VWCE_alloc_geo.rename(index = {'South korea': 'Korea'})

if "Russian Federation" in XMME_alloc_geo.index:
    XMME_alloc_geo = XMME_alloc_geo.rename(index = {"Russian Federation": 'Russia'})
if "Russian Federation" in SWDA_alloc_geo.index:
    SWDA_alloc_geo = SWDA_alloc_geo.rename(index = {"Russian Federation": 'Russia'})
if "Russian Federation" in VWCE_alloc_geo.index:
    VWCE_alloc_geo = VWCE_alloc_geo.rename(index = {"Russian Federation": 'Russia'})


In [16]:
df_etf = pd.DataFrame({
    'SWDA': SWDA_alloc_geo,
    'XMME': XMME_alloc_geo,
    'VWCE': VWCE_alloc_geo
}).fillna(0).sort_values( by = 'VWCE', ascending = False)
df_etf.round(3).head(10)

,SWDA,XMME,VWCE
United States,70.848,0.219,59.731
Japan,5.851,0.000,6.298
United Kingdom,3.890,0.000,3.596
China,0.000,23.467,3.151
Canada,3.505,0.047,3.071
Taiwan,0.000,22.858,2.753
Switzerland,2.344,0.184,2.241
Korea,0.000,0.000,2.226
France,2.611,0.000,2.138
Germany,2.293,0.000,2.089


Let $\alpha, \beta$ be the vectors of allocation of the SWDA and of the XMME etfs, and let $\gamma$ be the vector of allocation of the VWCE etf, we want to minimize the following function

$$
f\colon \mathbb R^2 \to \mathbb R\\
(x,y)\mapsto ||(\alpha, \beta)\cdot (x,y)^\top - \gamma||^2
$$

which is convex. We use a standard optimizator for this kind of function

In [17]:
# Define optimization problem and find the optimal weights using a standard optimization algorithm

alpha, beta, gamma = df_etf['SWDA'], df_etf['XMME'], df_etf['VWCE']
history = []

def objective(vars):
    x, y = vars
    val = np.linalg.norm(alpha * x + beta * y - gamma)**2
    return val

def callback(xk):
    # Calculate the function value at the current iteration's coordinates
    score = objective(xk)
    history.append(score)
    print(f"Iteration {len(history)}: {score}")

x0 = [0.5, 0.5]
cons = ({'type': 'eq', 'fun': lambda x: x[0] + x[1] - 1})

res = minimize(objective, x0, method='SLSQP', callback=callback, constraints=cons, options={'maxiter': 5000, 'disp': True, 'acc': 1e-3})
optimal_weights = res.x

C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\2890033822.py:20: OptimizeWarning: Unknown solver options: acc
  res = minimize(objective, x0, method='SLSQP', callback=callback, constraints=cons, options={'maxiter': 5000, 'disp': True, 'acc': 1e-3})


Iteration 1: 37395550449.02024
Iteration 2: 14.126840927988308
Optimization terminated successfully    (Exit mode 0)
            Current function value: 14.126840927988308
            Iterations: 2
            Function evaluations: 10
            Gradient evaluations: 2


In [18]:
# now we can look at the differences between the two portfolios and see how they can be combined to get a more balanced allocation.
balanced_allocation = optimal_weights[0] * df_etf['SWDA'] + optimal_weights[1] * df_etf['XMME']

comparison_df = pd.DataFrame({
    'SWDA+XMME': balanced_allocation,
    'VWCE': df_etf['VWCE']
})

comparison_df.round(3).sort_values(by = 'VWCE', ascending=False ).head(10)

,SWDA+XMME,VWCE
United States,60.718,59.731
Japan,5.012,6.298
United Kingdom,3.332,3.596
China,3.366,3.151
Canada,3.009,3.071
Taiwan,3.278,2.753
Switzerland,2.034,2.241
Korea,0.000,2.226
France,2.236,2.138
Germany,1.964,2.089


In [19]:
print(optimal_weights)

[0.85657865 0.14342135]


In [ ]:
# authentication to google APIs
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name("D:\giaco\Documents\project\sheet_edit_bot\my-project-giacomot-5913080db8bc.json", scope)
client = gspread.authorize(creds)


spreadsheet = client.open("SWDA-XMME_periodic_rebalancing")
sheet = spreadsheet.sheet1 


<>:4: SyntaxWarning: invalid escape sequence '\g'
<>:4: SyntaxWarning: invalid escape sequence '\g'
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\474498040.py:4: SyntaxWarning: invalid escape sequence '\g'
  creds = ServiceAccountCredentials.from_json_keyfile_name("D:\giaco\Documents\project\sheet_edit_bot\my-project-giacomot-5913080db8bc.json", scope)


{'spreadsheetId': '1bBrqZe3lpPk7-80an_ueNTqEgqLV8q1CvGjPSfj28J8',
 'updatedRange': 'Foglio1!A6',
 'updatedRows': 1,
 'updatedColumns': 1,
 'updatedCells': 1}

In [ ]:
# Update optimal weights in the sheet
sheet.update_acell('A2', optimal_weights[0])
sheet.update_acell('B2', optimal_weights[1])

# Update geographical allocations in the sheet
comparison_df = comparison_df.sort_values(by = 'VWCE', ascending=False)
N = comparison_df.shape[0]

country_list = comparison_df.index.tolist()
country_list = [[x] for x in country_list]

balanced_portfolio = comparison_df['SWDA+XMME'].values.tolist()
balanced_portfolio = [[x] for x in balanced_portfolio]

target_portfolio = comparison_df['VWCE'].values.tolist()
target_portfolio = [[x] for x in target_portfolio]


cell_interval = 'D2:D' + str(N+3)
sheet.update(country_list, cell_interval)

cell_interval = 'E2:E' + str(N+3)
sheet.update(balanced_portfolio, cell_interval)

cell_interval = 'F2:F' + str(N+3)
sheet.update(target_portfolio, cell_interval)


C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\326031935.py:20: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(cell_interval, country_list)
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\326031935.py:23: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(cell_interval, balanced_portfolio)
C:\Users\giaco\AppData\Local\Temp\ipykernel_26856\326031935.py:26: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(cell_interval, target_portfolio)


{'spreadsheetId': '1bBrqZe3lpPk7-80an_ueNTqEgqLV8q1CvGjPSfj28J8',
 'updatedRange': 'Foglio1!F2:F57',
 'updatedRows': 56,
 'updatedColumns': 1,
 'updatedCells': 56}